# Mahabharata KG: confidence-aware entity normalization on Kaggle

Implements *Entity Normalization in a Mahābhārata Knowledge Graph*: every PERSON mention gets candidate canonical entities and a confidence

`confidence = 0.30·A + 0.25·C + 0.20·R + 0.15·S + 0.10·E`

(A alias match, C context similarity, R relationship consistency, S source evidence, E entity type). It then applies the document's bands: ≥ 0.90 auto-accept, 0.75–0.89 accept with sample verification, 0.50–0.74 human review, < 0.50 reject.

It runs Phases 0–2 of the pipeline (chapter spine, mentions, appellations), then `src/phase2b_confidence.py`. No coreference or relation extraction runs, so this takes minutes, not hours.

## Inputs (Add Data)

The same private dataset as the full-pipeline notebook. The notebook searches all of `/kaggle/input` by name:

| what | used for |
|---|---|
| the repo code: `src/`, `out/`, `ontology/`, `annotations/` | the pipeline scripts; `out/relations.ttl` for relationship consistency and the before/after gate table |
| `Mahabharata LLM annotated/` (18 CSVs) | mentions, all parvas |
| `Mahabharat_Human_Annotated_Data/` (8 CSVs) | mentions, the human-checked parvas |
| `parva_pdfs/` (the parva PDFs) | chapter boundaries, for `(parva, adhyaya)` citations |
| `mahabharata_ner_by_sentence_master (2).csv` | sentence order and the corpus `canonical_id` prior |
| `mahabharata_ner_entities_master (2).csv` | checked by the preflight only |
| **optional:** `er_gold.csv` | your filled-in gold sheet (second run) |

## Settings

- **Accelerator:** GPU T4 is best, for the sentence encoder (about 1 minute on GPU, 10–15 on CPU). CPU works.
- **Internet: On**, for `pip` and the encoder download. With Internet off, set `ENCODER = "tfidf"` in Cell 5.
- Keep the dataset and notebook **private**: the corpus and the outputs carry verbatim copyrighted text.

## Two runs

1. **First run:** there's no gold file. You get the weighted scores, the review queue, and `gold_template.csv` (1,000 mentions for you to link).
2. Fill `correct_entity` in `gold_template.csv`, save it as `er_gold.csv`, upload it as a dataset and attach it. **Second run:** precision, recall and F1 per band, calibration (ECE and a reliability table), and weights learned by logistic regression.

In [ ]:
# Cell 1: find the inputs and assemble a writable working copy.
# /kaggle/input is read-only; the pipeline writes build/ next to src/, so the code
# is copied to /kaggle/working and the corpus is symlinked, not copied.
import os, shutil, sys
from pathlib import Path

INPUT = Path(os.environ.get("MBH_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("MBH_WORK", "/kaggle/working/mahabharata-main"))

LLM_OUTER = "Mahabharata LLM annotated-20260801T163642Z-1-001"
LLM_INNER = "Mahabharata LLM annotated"
HUM_OUTER = "Mahabharat_Human_Annotated_Data-20260801T163709Z-1-001"
HUM_INNER = "Mahabharat_Human_Annotated_Data"
MASTERS = ["mahabharata_ner_by_sentence_master (2).csv",
           "mahabharata_ner_entities_master (2).csv"]
CODE_DIRS = ["src", "ontology", "annotations", "out"]

def first(paths, what):
    paths = sorted(paths, key=lambda p: len(p.parts))
    if not paths:
        return None
    if len(paths) > 1:
        print("  note: %d copies of %s, using %s" % (len(paths), what, paths[0]))
    return paths[0]

repo = first([p.parent.parent for p in INPUT.rglob("phase1_mentions.py") if p.parent.name == "src"], "the repo")
if repo is None:
    raise SystemExit("No src/phase1_mentions.py under %s. Attach the dataset holding the repo code." % INPUT)
print("repo code:", repo)

WORK.mkdir(parents=True, exist_ok=True)
for d in CODE_DIRS:
    if (repo / d).is_dir():
        shutil.copytree(repo / d, WORK / d, dirs_exist_ok=True)

def link(target, link_path):
    link_path.parent.mkdir(parents=True, exist_ok=True)
    if link_path.is_symlink():
        link_path.unlink()
    elif link_path.exists():
        return  # a real copy already sits here; leave it
    link_path.symlink_to(target, target_is_directory=target.is_dir())

found = {}
for outer, inner in [(LLM_OUTER, LLM_INNER), (HUM_OUTER, HUM_INNER)]:
    hit = first([p for p in INPUT.rglob(inner) if p.is_dir()], inner)
    found[inner] = hit
    if hit:
        link(hit, WORK / outer / inner)

pdf = first([p.parent for p in INPUT.rglob("01_Adi.pdf")], "parva_pdfs")
found["parva_pdfs"] = pdf
if pdf:
    link(pdf, WORK / "parva_pdfs")

for name in MASTERS:
    # Tolerate a renamed upload, e.g. Kaggle dropping the " (2)".
    stem = name.split(" (")[0].replace(".csv", "")
    hit = first([p for p in INPUT.rglob(stem + "*.csv")], name)
    found[name] = hit
    if hit:
        link(hit, WORK / name)

# Optional gold standard for the second run.
gold = first([p for p in INPUT.rglob("er_gold*.csv")], "er_gold.csv")
found["er_gold.csv (optional)"] = gold
if gold:
    (WORK / "annotations").mkdir(exist_ok=True)
    shutil.copy2(gold, WORK / "annotations/er_gold.csv")

for k, v in found.items():
    print("%-45s %s" % (k, v or ("not attached" if "optional" in k else "MISSING")))

In [ ]:
# Cell 2: preflight. Every input Phases 0 to 2 open, checked before anything runs.
import re
src = (WORK / "src/phase1_mentions.py").read_text(encoding="utf-8")
llm = re.findall(r'"\d\d_\w+": "([^"]+\.csv)"', src.split("HUM_FILES")[0])
hum = re.findall(r'"\d\d_\w+": "([^"]+\.csv)"', src.split("HUM_FILES")[1])
need = ([WORK / LLM_OUTER / LLM_INNER / f for f in llm] +
        [WORK / HUM_OUTER / HUM_INNER / f for f in hum] +
        [WORK / "parva_pdfs" / ("%s.pdf" % p) for p in sorted(set(re.findall(r'"(\d\d_\w+)": "', src)))] +
        [WORK / m for m in MASTERS])
missing = [p for p in need if not p.exists()]
print("checked %d inputs: %d LLM CSVs, %d human CSVs, parva PDFs, 2 master CSVs" % (len(need), len(llm), len(hum)))
if missing:
    for p in missing:
        print("  MISSING", p.relative_to(WORK))
    raise SystemExit("%d required inputs are missing. Nothing has been run." % len(missing))
print("all inputs present")
print("out/relations.ttl:", "present" if (WORK / "out/relations.ttl").exists()
      else "MISSING: R falls back to co-mentions only and the gate table is empty")

In [ ]:
# Cell 3: dependencies. pandas, pyarrow, numpy, scikit-learn and torch ship with Kaggle.
import subprocess, sys
def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)
pip("rapidfuzz==3.14.6", "pymupdf", "sentence-transformers")

In [ ]:
%%writefile /kaggle/working/mahabharata-main/src/phase2b_confidence.py
"""Phase 2b. Confidence-aware entity resolution.

Implements the method in "Entity Normalization in a Mahabharata Knowledge Graph"
(the confidence-score design document) on top of the Phase 1 mention layer and
the Phase 2 appellation table. Every PERSON mention gets a candidate set, five
features per candidate, and a confidence:

    confidence = 0.30 A + 0.25 C + 0.20 R + 0.15 S + 0.10 E

  A  alias match. 1.0 if a dictionary links the surface to the candidate, 0.7 for
     an approximate spelling only, as the document specifies.
  C  context. Cosine between the mention's sentence and the candidate's profile
     (sentences where the candidate is named unambiguously), with the mention
     masked in both so the name cannot vote for itself. Raw cosines from a
     sentence encoder sit in a narrow band, so C is the cosine's percentile among
     the self-similarities of unambiguous mentions: 0.5 is a typical correct match.
  R  relationship consistency. Of the unambiguous names in the surrounding
     sentences, the share that are graph neighbours of the candidate
     (co-mention graph plus out/relations.ttl). 0.5 when no such name exists,
     because no evidence is not evidence against.
  S  source. The most reliable source proposing the link, on the document's scale:
     1.0 the text itself (identity, or Phase 2 asserted merge), 0.8 the seed
     alias list, 0.6 automatic extraction (apposition-only, corpus prior, spelling).
  E  entity type. 1 if the candidate is mostly labelled PERSON, else 0.

Bands follow the document: >= 0.90 auto-accept, 0.75 to 0.89 accept with sample
verification, 0.50 to 0.74 human review, below 0.50 reject. One addition: where
the runner-up is within MARGIN of the winner, the mention goes to human review
whatever its score. "Kounteya" can be any of four sons of Kunti, and a close
second is exactly the case the hard rule against guessing a link is about.

Nothing here writes to out/. Every link is a proposal in build/er/, and a merge
into the graph would need the decision recorded in docs/DECISIONS.md first.

Gold standard: if annotations/er_gold.csv exists (mention_id, correct_entity),
the script also reports precision, recall and F1 per band, calibration (ECE and a
reliability table), and learns the weights with logistic regression, grouped by
surface so no alias is in both train and test.

Run: python src/phase2b_confidence.py [--encoder tfidf|MODEL_NAME]
"""
import argparse, collections, json, os, re, sys, unicodedata

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phase2_entities import TITLES, norm_surface  # noqa: E402
from vocab import NOT_A_NAME  # noqa: E402

W = {"A": 0.30, "C": 0.25, "R": 0.20, "S": 0.15, "E": 0.10}
BANDS = [(0.90, "very_high", "auto_accept"), (0.75, "high", "accept_sample_verify"),
         (0.50, "medium", "human_review"), (0.0, "low", "reject")]
MARGIN = 0.10         # runner-up this close sends the mention to review
FUZZY_MIN = 90        # rapidfuzz ratio for a spelling variant
FUZZY_DOMINANCE = 3   # the variant must be this many times more frequent
PROFILE_K = 300       # sentences per candidate profile
MIN_COUNT = 3         # a surface seen fewer times is not a canonical entity
WINDOW = 2            # sentences either side for R
SEED = 20260930
OUT = "build/er"
GOLD = "annotations/er_gold.csv"

# Seed alias list, from general knowledge of the epic, in this translation's
# spelling. It is a secondary source (S = 0.8) and needs a human pass: it says
# which bearers an epithet CAN have, never which one a passage means. "self"
# keeps the surface as its own candidate, for names another person also bears
# (Bhima the king of Vidarbha, Vasudeva the father of Krishna).
SEED_ALIASES = {
    "Partha": ["Arjuna", "Yudhishthira", "Bhimasena"],
    "Parthas": ["Arjuna", "Yudhishthira", "Bhimasena"],
    "Kounteya": ["Arjuna", "Yudhishthira", "Bhimasena", "Karna"],
    "Kunti son": ["Arjuna", "Yudhishthira", "Bhimasena", "Karna"],
    "Dhananjaya": ["Arjuna"], "Phalguna": ["Arjuna"], "Savyasachi": ["Arjuna"],
    "Bibhatsu": ["Arjuna"], "Kiriti": ["Arjuna"], "Jishnu": ["Arjuna"],
    "Gudakesha": ["Arjuna"], "Vijaya": ["Arjuna", "self"], "Nara": ["Arjuna", "self"],
    "Dharmaraja": ["Yudhishthira", "Yama"], "Ajatashatru": ["Yudhishthira"],
    "Kouravya": ["Yudhishthira", "Dhritarashtra", "Duryodhana", "Bhishma"],
    "Bhima": ["Bhimasena", "self"], "Vrikodara": ["Bhimasena"],
    "Vasudeva": ["Krishna", "self"], "Keshava": ["Krishna"], "Janardana": ["Krishna"],
    "Madhava": ["Krishna", "self"], "Govinda": ["Krishna"], "Madhusudana": ["Krishna"],
    "Hrishikesha": ["Krishna"], "Achyuta": ["Krishna", "Vishnu"],
    "Varshneya": ["Krishna", "Satyaki", "self"], "Dasharha": ["Krishna", "self"],
    "Pundarikaksha": ["Krishna", "Vishnu"], "Shouri": ["Krishna"],
    "Hari": ["Krishna", "Vishnu"], "Narayana": ["Vishnu", "Krishna"],
    "Purushottama": ["Krishna", "Vishnu"],
    "Krishna": ["self", "Droupadi", "Vyasa"],
    "Krishna Dvaipayana": ["Vyasa"], "Dvaipayana": ["Vyasa"],
    "Radheya": ["Karna"], "Vaikartana": ["Karna"], "Suyodhana": ["Duryodhana"],
    "Bharadvaja": ["Drona", "self"], "Bharadvaja son": ["Drona"],
    "Drona son": ["Ashvatthama"], "Drouni": ["Ashvatthama"],
    "Gangeya": ["Bhishma"], "Devavrata": ["Bhishma"], "Shantanu son": ["Bhishma"],
    "Yuyudhana": ["Satyaki"], "Shini descendant": ["Satyaki"], "Shaineya": ["Satyaki"],
    "Panchali": ["Droupadi"], "Yajnaseni": ["Droupadi"], "Draupadi": ["Droupadi"],
    "Shakra": ["Indra"], "Vasava": ["Indra"], "Shatakratu": ["Indra"],
    "Purandara": ["Indra"], "Maghavan": ["Indra"],
    "Parshata": ["Dhrishtadyumna", "Drupada"], "Saindhava": ["Jayadratha"],
    "Soubala": ["Shakuni"], "Subhadra son": ["Abhimanyu"],
    "Goutama": ["Kripa", "self"], "Hardikya": ["Kritavarma"], "Kshatta": ["Vidura"],
    "Pritha": ["Kunti"], "Mahadeva": ["Shiva"], "Maheshvara": ["Shiva"],
    "Shankara": ["Shiva"], "Sharva": ["Shiva"], "Hara": ["Shiva"], "Ishana": ["Shiva"],
    "Sthanu": ["Shiva"], "Bhava": ["Shiva"], "Rudra": ["Shiva", "self"],
    "Ushanas": ["Shukra"], "Vaishravana": ["Kubera"], "Svayambhu": ["Brahma"],
    "Prajapati": ["Brahma", "self"], "Kumara": ["Skanda", "self"],
    "Bhargava": ["Parashurama", "Shukra", "self"], "Rama": ["self", "Parashurama", "Balarama"],
    "Souti": ["Ugrashrava"],
}
# Where a "self" reading of a seed alias can apply, as (parva, first adhyaya,
# last adhyaya). Outside these chapters the name is only its seed candidates.
# King Bhima of Vidarbha is the Nala tale; elsewhere "Bhima" is the Pandava, and
# the bare name, which can never have a context profile, won C = 0.5 by default
# on 1,007 of 1,517 mentions. Same standing as the seed list: needs a human pass.
SELF_SCOPE = {"Bhima": [("03_Aranyaka", 50, 78)]}
S_SOURCE = {"identity": 1.0, "phase2_asserted": 1.0, "seed": 0.8,
            "phase2_apposition": 0.6, "prior": 0.6, "spelling": 0.6}


def fold(s):
    """Orthographic key: the Phase 2 normalization plus diacritics, so the
    document's Partha with a macron and the translation's Partha meet."""
    s = unicodedata.normalize("NFKD", str(s))
    return norm_surface("".join(c for c in s if not unicodedata.combining(c)))


def clean_surface(s):
    toks = [t for t in str(s).split() if t not in TITLES]
    return " ".join(toks)


def band(c):
    for lo, name, action in BANDS:
        if c >= lo:
            return name, action


# ---------------------------------------------------------------- candidates

def load_mentions():
    df = pd.read_parquet("build/mentions.parquet")
    df = df[df.label == "PERSON"].copy()
    df["clean"] = df.surface.map(clean_surface)
    df = df[df.clean.str.fullmatch(r"[A-Z][a-z]{2,}( [A-Za-z][a-z]+)*")
            & ~df.clean.isin(NOT_A_NAME)]
    df["key"] = df.clean.map(fold)
    return df.reset_index(drop=True)


def all_labels():
    """Majority label per folded surface over every mention, not only PERSON."""
    df = pd.read_parquet("build/mentions.parquet", columns=["surface", "label"])
    df["key"] = df.surface.map(clean_surface).map(fold)
    return df.groupby("key").label.agg(lambda x: x.value_counts().idxmax()).to_dict()


def build_dictionary(men):
    """(folded alias -> {folded candidate: [sources]}, display names, canonical set)."""
    counts = men.key.value_counts()
    display = men.groupby("key").clean.agg(lambda x: x.value_counts().idxmax()).to_dict()
    links = collections.defaultdict(lambda: collections.defaultdict(set))
    no_self = set()

    for alias, cands in SEED_ALIASES.items():
        a = fold(alias)
        for c in cands:
            links[a][a if c == "self" else fold(c)].add("seed")
        if "self" not in cands:
            no_self.add(a)

    # Spelling variants and drop-cap truncations ("udhishthira", "hishma"). The
    # minority form stops being an entity of its own and gets A = 0.7.
    from rapidfuzz import fuzz, process
    frequent = [k for k, n in counts.items() if n >= MIN_COUNT and " " not in k]
    for k in frequent:
        if len(k) < 5:
            continue
        for t, score, _ in process.extract(k, frequent, scorer=fuzz.ratio,
                                           score_cutoff=FUZZY_MIN, limit=4):
            if t == k or t == k + "s" or k == t + "s":
                continue
            if counts[t] >= FUZZY_DOMINANCE * counts[k]:
                links[k][t].add("spelling")
                no_self.add(k)

    # Phase 2. An asserted merge (apposition and corpus prior agree) and a prior
    # rewrite each propose a candidate. Apposition evidence alone only supports a
    # candidate something else proposed: adjacency also catches lists of names,
    # and Phase 2's direction rule reads "Partha Arjuna" as Arjuna being an
    # epithet of Partha, which made every plain "Arjuna" a tie.
    if os.path.exists("build/phase2_resolution.json"):
        res = json.load(open("build/phase2_resolution.json", encoding="utf-8"))
        for dep, r in res.items():
            d = fold(dep)
            if r.get("prior"):
                links[d][fold(r["prior"])].add("prior")
            for c in r["candidates"]:
                if not c["entity"]:
                    continue
                e = fold(c["entity"])
                if r["status"] == "asserted":
                    links[d][e].add("phase2_asserted")
                elif r["status"] == "prior_only":
                    links[d][e].add("prior")
                elif e in links.get(d, {}):
                    links[d][e].add("phase2_apposition")
            if r["status"] == "asserted":
                no_self.add(d)

    # A candidate the seed list itself resolves to the alias, or to another of
    # its candidates, is that person under a second name, not a rival. The corpus
    # prior calls the Pandava "Bhima" while the seed makes Bhima an epithet of
    # Bhimasena, so each proposed the other: neither was ever unambiguous, neither
    # got a context profile, every mention of either tied at C = 0.5, and the free
    # 0.5 then outscored Arjuna's real C on "Partha". The link folds into the
    # candidate it names. A seed link is never folded: the seed is the specific word.
    seed_to = {fold(a): {fold(c) for c in cs if c != "self"} for a, cs in SEED_ALIASES.items()}
    for a in list(links):
        for c in list(links[a]):
            if c == a or "seed" in links[a][c]:
                continue
            same = seed_to.get(c, set()) & (set(links[a]) | {a})
            if same:
                for d in same - {a}:
                    links[a][d] |= links[a][c]
                del links[a][c]

    canonical ={k for k, n in counts.items() if n >= MIN_COUNT and k not in no_self}
    for a in list(links):
        for c in list(links[a]):
            if c != a and c not in counts:          # candidate never named in this text
                del links[a][c]
    return links, display, canonical


def candidates_for(key, links, canonical):
    cands = {}
    if key in canonical:
        # A name the seed list says someone else also bears ("Bhima") is not
        # identity evidence for the unmerged bearer. It enters at the seed's level.
        cands[key] = {"seed"} if "seed" in links.get(key, {}).get(key, ()) else {"identity"}
    for c, src in links.get(key, {}).items():
        if c == key and key not in canonical:
            continue
        cands.setdefault(c, set()).update(src)
    # A compound whose every token points at one candidate names it outright:
    # "Dharmaraja Yudhishthira". "Drona son" does not, and its tokens disagree.
    toks = key.split()
    if not cands and len(toks) > 1:
        sets = [set(candidates_for(t, links, canonical)) for t in toks]
        common = set.intersection(*sets) if all(sets) else set()
        for c in common:
            cands[c] = {"phase2_apposition"}
    return cands


# ---------------------------------------------------------------- features

def mask(sentence, surfaces):
    for s in sorted(surfaces, key=len, reverse=True):
        sentence = re.sub(r"\b%s\b" % re.escape(s), "someone", sentence)
    return sentence


class Encoder:
    def __init__(self, name):
        self.name = name
        if name == "tfidf":
            from sklearn.feature_extraction.text import TfidfVectorizer
            self.vec = None
            self.TfidfVectorizer = TfidfVectorizer
        else:
            from sentence_transformers import SentenceTransformer
            import torch
            dev = "cuda" if torch.cuda.is_available() else "cpu"
            print("encoder %s on %s" % (name, dev))
            self.model = SentenceTransformer(name, device=dev)

    def fit(self, texts):
        if self.name == "tfidf":
            self.vec = self.TfidfVectorizer(min_df=2, sublinear_tf=True,
                                            stop_words="english").fit(texts)

    def __call__(self, texts):
        if self.name == "tfidf":
            from sklearn.preprocessing import normalize
            return normalize(self.vec.transform(texts)).astype(np.float32)
        return self.model.encode(list(texts), batch_size=256, normalize_embeddings=True,
                                 show_progress_bar=True, convert_to_numpy=True)


def dense(x):
    return x.toarray() if hasattr(x, "toarray") else x


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--encoder", default="sentence-transformers/multi-qa-MiniLM-L6-cos-v1")
    args = ap.parse_args()
    os.makedirs(OUT, exist_ok=True)
    rng = np.random.default_rng(SEED)

    men = load_mentions()
    labels = all_labels()
    links, display, canonical = build_dictionary(men)
    men["cands"] = [candidates_for(k, links, canonical) for k in men.key]
    scope = {fold(a): spans for a, spans in SELF_SCOPE.items()}

    def in_scope(r):
        spans = scope.get(r.key)
        if not spans or r.key not in r.cands or len(r.cands) == 1:
            return r.cands
        if any(r.parva == p and lo <= r.adhyaya_ordinal <= hi for p, lo, hi in spans):
            return r.cands
        return {c: s for c, s in r.cands.items() if c != r.key}
    men["cands"] = [in_scope(r) for r in men.itertuples()]
    men = men[men.cands.map(len) > 0].reset_index(drop=True)
    men["n_cands"] = men.cands.map(len)
    print("PERSON mentions with candidates: %d, canonical entities: %d, aliases: %d"
          % (len(men), len(canonical), sum(1 for k in links if k not in canonical)))

    # Unambiguous mentions: the name is its own and only candidate.
    unamb = men[(men.n_cands == 1) & men.apply(lambda r: r.key in r.cands, axis=1)]

    # --- C: profiles and mention contexts
    enc = Encoder(args.encoder)
    prof_rows = []
    for k, g in unamb.groupby("key"):
        take = g.sample(min(len(g), PROFILE_K), random_state=SEED)
        prof_rows += [(k, mask(s, [c])) for s, c in zip(take.sentence, take.clean)]
    prof = pd.DataFrame(prof_rows, columns=["key", "text"])
    men["ctx"] = [mask(s, [c]) for s, c in zip(men.sentence, men.clean)]
    enc.fit(pd.concat([prof.text, men.ctx]).tolist())
    pv = dense(enc(prof.text.tolist()))
    centroid = {}
    for k, idx in prof.groupby("key").indices.items():
        v = pv[idx].mean(axis=0)
        centroid[k] = v / (np.linalg.norm(v) or 1.0)
    mv = dense(enc(men.ctx.tolist()))

    def cos(i, k):
        return float(mv[i] @ centroid[k]) if k in centroid else np.nan

    # Calibrate C against the self-similarity of unambiguous mentions.
    self_sims = np.array(sorted(cos(i, k) for i, k in zip(unamb.index, unamb.key)
                                if k in centroid))
    to_c = lambda x: 0.5 if np.isnan(x) else float(np.searchsorted(self_sims, x) / len(self_sims))

    # --- R: neighbour graph from unambiguous co-mentions and the asserted graph
    sent_names = unamb.groupby(["parva", "sent_id"]).key.agg(set)
    co = collections.Counter()
    for ns in sent_names:
        for a in ns:
            for b in ns:
                if a < b:
                    co[(a, b)] += 1
    nbr = collections.defaultdict(set)
    for (a, b), n in co.items():
        if n >= 2:
            nbr[a].add(b)
            nbr[b].add(a)
    triples = read_relations("out/relations.ttl")
    for s, _, o, _ in triples:
        nbr[fold(s)].add(fold(o))
        nbr[fold(o)].add(fold(s))
    sn_index = sent_names.to_dict()

    def context_names(parva, sid):
        out = set()
        for d in range(-WINDOW, WINDOW + 1):
            out |= sn_index.get((parva, sid + d), set())
        return out

    # --- score every (mention, candidate)
    rows = []
    for i, r in enumerate(men.itertuples()):
        near = context_names(r.parva, r.sent_id) - set(r.cands) - {r.key}
        for c, src in r.cands.items():
            A = 0.7 if src == {"spelling"} else 1.0
            C = to_c(cos(i, c))
            R = len(near & nbr[c]) / len(near) if near else 0.5
            S = max(S_SOURCE[s] for s in src)
            E = 1.0 if labels.get(c) == "PERSON" else 0.0
            conf = W["A"] * A + W["C"] * C + W["R"] * R + W["S"] * S + W["E"] * E
            rows.append((r.mention_id, c, A, C, R, S, E, conf, "+".join(sorted(src))))
    pairs = pd.DataFrame(rows, columns=["mention_id", "candidate", "A", "C", "R", "S", "E",
                                        "confidence", "sources"])
    pairs["candidate"] = pairs.candidate.map(lambda k: display.get(k, k))
    links_df = decide(pairs, men, display)
    page = write_outputs(pairs, links_df, men, display, triples, rng)
    gold = None
    if os.path.exists(GOLD):
        gold = evaluate(pairs, links_df)
    else:
        print("%s not found: no precision, recall, calibration or learned weights yet."
              " Fill %s/gold_template.csv and install it there." % (GOLD, OUT))
    write_page(pairs, links_df, page, gold)


def decide(pairs, men, display):
    pairs = pairs.sort_values(["mention_id", "confidence"], ascending=[True, False])
    top = pairs.groupby("mention_id").head(2)
    first = top.groupby("mention_id").first()
    second = top.groupby("mention_id").nth(1).set_index("mention_id")
    out = men.set_index("mention_id")[["parva", "sent_id", "adhyaya_ordinal", "tier",
                                        "surface", "sentence", "n_cands"]].join(first)
    out["runner_up"] = second.candidate
    out["margin"] = (out.confidence - second.confidence).fillna(1.0)
    b = out.confidence.map(band)
    out["band"] = [x[0] for x in b]
    out["action"] = [x[1] for x in b]
    close = (out.margin < MARGIN) & out.action.isin(["auto_accept", "accept_sample_verify"])
    out.loc[close, "action"] = "human_review"
    out["close_runner_up"] = close
    return out.reset_index()


# ---------------------------------------------------------------- graph effect

_TRIPLE = re.compile(r"mbh:subject <[^>]*/e/([^>]+)> ; mbh:predicate mbh:(\w+) ; "
                     r"mbh:object <[^>]*/e/([^>]+)>.*?mbh:inPassage <[^>]*/p/([^>]+)>")


def read_relations(path):
    if not os.path.exists(path):
        return []
    return [(s.replace("_", " "), p, o.replace("_", " "), psg)
            for s, p, o, psg in _TRIPLE.findall(open(path, encoding="utf-8").read())]


def gate(triples):
    t = {(s, p, o) for s, p, o in triples}
    deg = collections.Counter()
    for s, _, o in t:
        deg[s] += 1
        deg[o] += 1
    share = collections.Counter(p for _, p, _ in t)
    top, n = share.most_common(1)[0] if share else ("-", 0)
    top_ent = deg.most_common(1)[0] if deg else ("-", 0)
    return {"triples": len(t), "entities": len(deg),
            "degree_ge_5": sum(1 for v in deg.values() if v >= 5),
            "max_degree": "%s %d" % top_ent,
            "mean_degree": round(sum(deg.values()) / max(len(deg), 1), 2),
            "top_relation": top, "top_share": round(n / max(len(t), 1), 4)}


ACCEPTED = ("auto_accept", "accept_sample_verify")


def endpoint_map(links_df):
    """A function (name, parva, sent_id, chapter) -> entity, from accepted links only.

    Most specific first: the accepted link of that name in that sentence, then
    the chapter where every accepted mention of the name agrees on one entity,
    then corpus-wide, but only for a name with a single candidate everywhere:
    fifteen accepted "Parshata" mentions agreeing on Dhrishtadyumna say nothing
    about the 149 that went to review and may be Drupada. Anything else keeps
    the name it was written with."""
    lf = links_df.assign(s=links_df.surface.map(clean_surface))
    acc = lf[lf.action.isin(ACCEPTED)]
    acc = acc.assign(ch=acc.parva + "/" + acc.adhyaya_ordinal.astype("Int64").astype(str))
    unique = lambda g: {k: next(iter(v)) for k, v in g.candidate.agg(set).items() if len(v) == 1}
    by_sent = unique(acc.groupby(["s", "parva", "sent_id"]))
    by_ch = unique(acc.groupby(["s", "ch"]))
    single = lf.groupby("s").n_cands.max() == 1
    glob = unique(acc[acc.s.map(single)].groupby("s"))

    def m(name, parva=None, sent_id=None, chapter=None):
        if sent_id is not None and (name, parva, sent_id) in by_sent:
            return by_sent[(name, parva, sent_id)]
        return by_ch.get((name, chapter), glob.get(name, name))
    return m


def remap(triples, links_df):
    """Relation endpoints of out/relations.ttl rewritten through accepted links.
    A ttl triple carries a chapter and no sentence, so it maps at chapter level."""
    m = endpoint_map(links_df)
    out = []
    for s, p, o, psg in triples:
        s2, o2 = m(s, chapter=psg), m(o, chapter=psg)
        if s2 != o2:
            out.append((s2, p, o2))
    return out


# ---------------------------------------------------------------- outputs

def write_outputs(pairs, links_df, men, display, triples, rng):
    pairs.to_csv(os.path.join(OUT, "candidate_features.csv"), index=False)
    cols = ["mention_id", "parva", "sent_id", "adhyaya_ordinal", "surface", "candidate", "confidence",
            "band", "action", "runner_up", "margin", "A", "C", "R", "S", "E", "sources",
            "n_cands", "sentence"]
    links_df[cols].to_csv(os.path.join(OUT, "mention_links.csv"), index=False)
    links_df[links_df.action == "human_review"][cols].to_csv(
        os.path.join(OUT, "review_queue.csv"), index=False)

    # Canonical store and alias table, from accepted links only.
    acc = links_df[links_df.action.isin(ACCEPTED)]
    ents = acc.candidate.value_counts()
    pid = {e: "P%04d" % (i + 1) for i, e in enumerate(ents.index)}
    alias = (acc.assign(alias=acc.surface.map(clean_surface))
             .groupby(["candidate", "alias"])
             .agg(mentions=("mention_id", "size"), mean_confidence=("confidence", "mean"),
                  sources=("sources", lambda x: ";".join(sorted(set("+".join(x).split("+"))))))
             .reset_index())
    alias.insert(0, "person_id", alias.candidate.map(pid))
    alias = alias.rename(columns={"candidate": "canonical_name"})
    alias["alias_type"] = ""       # patronymic, epithet, spelling: a human fills these
    alias["language"] = ""
    alias.sort_values(["person_id", "mentions"], ascending=[True, False]).to_csv(
        os.path.join(OUT, "alias_table.csv"), index=False)
    store = (alias.groupby(["person_id", "canonical_name"])
             .apply(lambda g: pd.Series({
                 "mentions": g.mentions.sum(),
                 "aliases": "; ".join(a for a in g.alias if a != g.name[1])}),
                    include_groups=False)
             .reset_index().sort_values("person_id"))
    store.to_csv(os.path.join(OUT, "canonical_entities.csv"), index=False)

    write_gold_template(links_df, pairs, rng)

    before, after = gate(triples and [t[:3] for t in triples]), gate(remap(triples, links_df))
    by_band = links_df.band.value_counts().reindex([b[1] for b in BANDS]).fillna(0).astype(int)
    by_action = links_df.action.value_counts()
    amb = links_df[links_df.n_cands > 1]
    L = ["# Phase 2b. Confidence-aware entity resolution", "",
         "Generated by `src/phase2b_confidence.py`. Proposals only: nothing here is"
         " written to `out/`, and the weights are the design document's, not learned,"
         " unless the gold-standard section below says otherwise.", "",
         "Confidence is not accuracy. Until `%s` exists these scores are uncalibrated." % GOLD,
         "", "## Mentions", "",
         "- PERSON mentions scored: **%d**" % len(links_df),
         "- with more than one candidate: **%d**" % len(amb),
         "- runner-up within %.2f, sent to review whatever the score: **%d**"
         % (MARGIN, int(links_df.close_runner_up.sum())), "",
         "| band | mentions |", "|---|---|"] + \
        ["| %s | %d |" % (k, v) for k, v in by_band.items()] + \
        ["", "| action | mentions |", "|---|---|"] + \
        ["| %s | %d |" % (k, v) for k, v in by_action.items()] + \
        ["", "## Ambiguous surfaces, how their mentions split", "",
         "| surface | mentions | accepted as | to review |", "|---|---|---|---|"]
    ambiguous = []
    for s, g in amb.groupby(amb.surface.map(clean_surface)):
        if len(g) < 20:
            continue
        a = g[g.action.isin(ACCEPTED)].candidate.value_counts()
        ambiguous.append({"surface": s, "mentions": len(g),
                          "accepted": {k: int(v) for k, v in a.items()},
                          "review": int((g.action == "human_review").sum())})
        L.append("| %s | %d | %s | %d |" % (s, len(g), ", ".join("%s %d" % kv for kv in a.items())
                                             or "-", (g.action == "human_review").sum()))
    L += ["", "## Effect on the Phase 6 entry gate", "",
          "`out/relations.ttl` with its endpoints rewritten through accepted links only.", "",
          "| measure | before | after | required |", "|---|---|---|---|",
          "| distinct triples | %d | %d | >= 10,000 |" % (before["triples"], after["triples"]),
          "| entities | %d | %d | |" % (before["entities"], after["entities"]),
          "| entities of degree >= 5 | %d | %d | >= 500 |" % (before["degree_ge_5"],
                                                             after["degree_ge_5"]),
          "| largest relation share | %.1f%% %s | %.1f%% %s | <= 40%% |"
          % (100 * before["top_share"], before["top_relation"],
             100 * after["top_share"], after["top_relation"]),
          "| mean degree | %.2f | %.2f | |" % (before["mean_degree"], after["mean_degree"]),
          "| highest degree | %s | %s | |" % (before["max_degree"], after["max_degree"]), "",
          "Read these with care. `out/relations.ttl` is not the parquet the gate reads, so"
          " the before column is this file's own count, not the README's. Normalization"
          " merges nodes: Shakra, Vasava and Maghavan become Indra. A triple written twice"
          " under two names becomes one, a triple between two names of one person"
          " becomes a self-loop and is dropped, and several nodes of degree 5 become one."
          " So the triple count and the degree-5 count can fall while each character's"
          " node gets deeper. Normalization cannot add a triple; the 10,000 and the 40%"
          " are extraction problems.", ""]
    open(os.path.join(OUT, "er_report.md"), "w", encoding="utf-8").write("\n".join(L))
    json.dump({"before": before, "after": after, "bands": by_band.to_dict(),
               "actions": by_action.to_dict(), "weights": W},
              open(os.path.join(OUT, "er_summary.json"), "w"), indent=1)
    print("\n".join(L))
    return {"before": before, "after": after, "ambiguous": ambiguous, "alias": alias,
            "summary": {"scored": len(links_df), "multi": len(amb),
                        "close": int(links_df.close_runner_up.sum()),
                        "bands": {k: int(v) for k, v in by_band.items()},
                        "actions": {k: int(v) for k, v in by_action.items()}}}


def write_gold_template(links_df, pairs, rng, n=1000, per_surface=15):
    """1,000 mentions for a human to link, as the document suggests. Stratified by
    band, at most 15 per surface so "Partha" cannot fill it. The system's answer
    is kept out of the sheet so the annotator is not anchored on it; it sits in
    gold_key.csv, which should not be opened before the sheet is done."""
    cand_lists = pairs.groupby("mention_id").candidate.agg(lambda x: "; ".join(sorted(x)))
    df = links_df.assign(s=links_df.surface.map(clean_surface))
    df = df.sample(frac=1, random_state=SEED).groupby("s").head(per_surface)
    take = df.groupby("band").head(n // len(BANDS))
    if len(take) < n:
        rest = df.drop(take.index)
        take = pd.concat([take, rest.sample(min(len(rest), n - len(take)), random_state=SEED)])
    take = take.sample(frac=1, random_state=SEED)
    sheet = pd.DataFrame({
        "mention_id": take.mention_id, "parva": take.parva, "adhyaya": take.adhyaya_ordinal,
        "surface": take.surface,
        "sentence": [s.replace(m, "[[" + m + "]]", 1) for s, m in zip(take.sentence, take.surface)],
        "candidates": take.mention_id.map(cand_lists),
        "correct_entity": ""})
    sheet.to_csv(os.path.join(OUT, "gold_template.csv"), index=False)
    take[["mention_id", "candidate", "confidence", "band", "action"]].to_csv(
        os.path.join(OUT, "gold_key.csv"), index=False)


# ---------------------------------------------------------------- gold standard

def evaluate(pairs, links_df):
    """P, R, F1 and calibration from annotations/er_gold.csv, then learned weights.

    correct_entity holds a candidate name, OTHER for a person outside the set, or
    NOT_A_PERSON. UNSURE and blank rows are dropped. A mention counts as predicted
    when the system accepts it; TP when the accepted entity is the gold one."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import GroupKFold

    gold = pd.read_csv(GOLD, dtype=str).dropna(subset=["correct_entity"])
    gold["correct_entity"] = gold.correct_entity.str.strip()
    gold = gold[~gold.correct_entity.str.upper().isin(["", "UNSURE"])]
    g = links_df.merge(gold[["mention_id", "correct_entity"]], on="mention_id")
    has_gold_entity = ~g.correct_entity.str.upper().isin(["OTHER", "NOT_A_PERSON"])
    data = {"n": len(g), "thresholds": [], "calibration": [], "coef": None}
    L = ["", "## Gold standard (%d mentions from %s)" % (len(g), GOLD), "",
         "| accept at | predicted | TP | FP | FN | precision | recall | F1 |",
         "|---|---|---|---|---|---|---|---|"]
    for t in (0.90, 0.75, 0.50):
        pred = g.confidence >= t
        tp = int((pred & (g.candidate == g.correct_entity)).sum())
        fp = int(pred.sum()) - tp
        fn = int(has_gold_entity.sum()) - tp
        p, r = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
        f1 = 2 * p * r / max(p + r, 1e-9)
        L.append("| %.2f | %d | %d | %d | %d | %.3f | %.3f | %.3f |"
                 % (t, pred.sum(), tp, fp, fn, p, r, f1))
        data["thresholds"].append([t, int(pred.sum()), tp, fp, fn, p, r, f1])
    y = (g.candidate == g.correct_entity).astype(int).values
    lines, cal = reliability("design weights", g.confidence.values, y)
    L += lines
    data["calibration"].append(cal)

    # Learned weights: one row per (mention, candidate), grouped by surface.
    pg = pairs.merge(gold[["mention_id", "correct_entity"]], on="mention_id")
    pg["y"] = (pg.candidate == pg.correct_entity).astype(int)
    pg["group"] = pg.mention_id.map(links_df.set_index("mention_id").surface.map(clean_surface))
    X = pg[list(W)].values
    oof = np.zeros(len(pg))
    folds = min(5, pg.group.nunique())
    if folds >= 2 and pg.y.nunique() == 2:
        for tr, te in GroupKFold(folds).split(X, pg.y, pg.group):
            if pg.y.iloc[tr].nunique() < 2:
                continue
            oof[te] = LogisticRegression(max_iter=1000).fit(X[tr], pg.y.iloc[tr]).predict_proba(X[te])[:, 1]
        full = LogisticRegression(max_iter=1000).fit(X, pg.y)
        coef = full.coef_[0]
        pg["p"] = oof
        top = pg.sort_values("p", ascending=False).groupby("mention_id").head(1)
        lines, cal = reliability("learned weights, out of fold", top.p.values, top.y.values)
        L += lines
        data["calibration"].append(cal)
        data["coef"] = []
        L += ["", "Learned coefficients (logistic regression, all gold rows):", "",
              "| feature | coefficient | share of positive weight | design weight |",
              "|---|---|---|---|"]
        pos = np.clip(coef, 0, None).sum() or 1.0
        for f, c in zip(W, coef):
            L.append("| %s | %.3f | %.2f | %.2f |" % (f, c, max(c, 0) / pos, W[f]))
            data["coef"].append([f, float(c), float(max(c, 0) / pos), W[f]])
        pairs["p_learned"] = full.predict_proba(pairs[list(W)].values)[:, 1]
        pairs.to_csv(os.path.join(OUT, "candidate_features_learned.csv"), index=False)
    else:
        L.append("\nToo few labelled surfaces or classes to learn weights.")
    open(os.path.join(OUT, "er_report.md"), "a", encoding="utf-8").write("\n".join(L) + "\n")
    print("\n".join(L))
    return data


def reliability(name, p, y, bins=10):
    """(report lines, the same numbers as data for the page)."""
    if len(p) == 0:
        return ["", "No rows for %s." % name], {"name": name, "ece": 0.0, "brier": 0.0, "rows": []}
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges) - 1, 0, bins - 1)
    ece = sum(abs(p[idx == b].mean() - y[idx == b].mean()) * (idx == b).mean()
              for b in range(bins) if (idx == b).any())
    L = ["", "Calibration, %s: ECE %.3f, Brier %.3f" % (name, ece, np.mean((p - y) ** 2)), "",
         "| confidence | mentions | mean confidence | share correct |", "|---|---|---|---|"]
    rows = []
    for b in range(bins):
        m = idx == b
        if m.any():
            L.append("| %.1f to %.1f | %d | %.3f | %.3f |"
                     % (edges[b], edges[b + 1], m.sum(), p[m].mean(), y[m].mean()))
            rows.append(["%.1f to %.1f" % (edges[b], edges[b + 1]), int(m.sum()),
                         float(p[m].mean()), float(y[m].mean())])
    return L, {"name": name, "ece": float(ece), "brier": float(np.mean((p - y) ** 2)), "rows": rows}


# ---------------------------------------------------------------- page

def write_page(pairs, links_df, page, gold):
    """build/er/er.html, er-data.js and explorer.css, the way out/index.html is
    built: a fixed page from src/er_page.html and its data as a .js file that
    assigns one global, so the folder opens from file:// with no server."""
    import shutil
    here = os.path.dirname(os.path.abspath(__file__))
    cands = {}
    for mid, g in pairs.sort_values("confidence", ascending=False).groupby("mention_id", sort=False):
        cands[mid] = [[c, round(f, 3), round(a, 3), round(cc, 3), round(r, 3), round(s, 3),
                       round(e, 3), src]
                      for c, f, a, cc, r, s, e, src in zip(g.candidate, g.confidence, g.A, g.C,
                                                           g.R, g.S, g.E, g.sources)]
    actions = [b[2] for b in BANDS]
    sentences, sidx, mentions = [], {}, []
    for r in links_df.itertuples():
        if r.sentence not in sidx:
            sidx[r.sentence] = len(sentences)
            sentences.append(r.sentence)
        ad = None if pd.isna(r.adhyaya_ordinal) else int(r.adhyaya_ordinal)
        mentions.append([r.mention_id, r.surface, r.parva, ad, sidx[r.sentence],
                         actions.index(r.action), cands[r.mention_id]])
    entities = []
    for (pid, name), g in page["alias"].groupby(["person_id", "canonical_name"]):
        g = g.sort_values("mentions", ascending=False)
        entities.append({"id": pid, "name": name, "mentions": int(g.mentions.sum()),
                         "aliases": [[a, int(n), round(float(c), 3), s] for a, n, c, s in
                                     zip(g.alias, g.mentions, g.mean_confidence, g.sources)]})
    entities.sort(key=lambda e: e["id"])
    data = {"weights": W, "bands": [[lo, n, a] for lo, n, a in BANDS], "actions": actions,
            "margin": MARGIN, "summary": page["summary"],
            "gate": {"before": page["before"], "after": page["after"]},
            "ambiguous": page["ambiguous"], "entities": entities,
            "sentences": sentences, "mentions": mentions, "gold": gold}
    with open(os.path.join(OUT, "er-data.js"), "w", encoding="utf-8") as f:
        f.write("window.KGER=")
        json.dump(data, f, separators=(",", ":"))
        f.write(";\n")
    shutil.copy(os.path.join(here, "er_page.html"), os.path.join(OUT, "er.html"))
    if os.path.exists("out/explorer.css"):
        shutil.copy("out/explorer.css", os.path.join(OUT, "explorer.css"))
    print("page: %s/er.html with er-data.js, %.1f MB, %d mentions, %d characters"
          % (OUT, os.path.getsize(os.path.join(OUT, "er-data.js")) / 1e6,
             len(mentions), len(entities)))
    write_standalone(here)


def write_standalone(here):
    """build/er/mahabharata_er.html: the same page as one file, stylesheet and data
    inlined, for when the three files would be separated (a single download from
    Kaggle's Output tab, an email attachment). "</" inside the data is escaped so
    a sentence can never close the script element early."""
    html = open(os.path.join(here, "er_page.html"), encoding="utf-8").read()
    css = (open("out/explorer.css", encoding="utf-8").read()
           if os.path.exists("out/explorer.css") else "")
    data = open(os.path.join(OUT, "er-data.js"), encoding="utf-8").read().replace("</", "<\\/")
    link = '<link rel="stylesheet" href="explorer.css">'
    script = '<script defer src="er-data.js"></script>'
    assert html.count(link) == 1 and html.count(script) == 1, "er_page.html changed shape"
    html = html.replace(link, "<style>\n" + css + "\n</style>")
    html = html.replace(script, "<script>\n" + data + "</script>")
    path = os.path.join(OUT, "mahabharata_er.html")
    open(path, "w", encoding="utf-8").write(html)
    print("single-file page: %s, %.1f MB" % (path, os.path.getsize(path) / 1e6))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/mahabharata-main/src/er_page.html
<!doctype html>
<html lang="en"><head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Mahabharata | Entity normalization</title>
<!-- Built like out/index.html: a fixed page, its data in a .js file that
     assigns one global, so it opens from file:// with no server and no request.
     src/phase2b_confidence.py copies this file next to er-data.js. -->
<link rel="stylesheet" href="explorer.css">
<script defer src="er-data.js"></script>
<style>
:root{--very_high:#216a4e;--high:#5c9a6f;--medium:#b58a3c;--low:#a2503f;--card:#fff}
body{background:#f5f7f2}
.topbar .wordmark span{font:600 10px/1.2 system-ui,sans-serif;letter-spacing:.14em;color:#9fb5a6}
.er-stats{display:flex;gap:26px;margin-left:auto;font-size:12px;color:#aebfb2;flex-wrap:wrap}
.er-stats b{display:block;font:22px/1.1 Georgia,serif;color:#fff}
.er-banner{padding:9px 28px;background:#fbf5e9;color:#7c6240;border-bottom:1px solid #e8dbc6;font-size:12px;line-height:1.6}
.er-tabs{display:flex;gap:4px;padding:12px 28px 0;background:#fff;border-bottom:1px solid var(--border);overflow-x:auto}
.er-tabs button{border:0;background:none;padding:10px 14px;font:inherit;font-size:13px;color:var(--muted);border-bottom:2px solid transparent;cursor:pointer;white-space:nowrap}
.er-tabs button[aria-selected=true]{color:var(--ink);border-bottom-color:var(--accent);font-weight:600}
.er-panel{max-width:1500px;margin:0 auto;padding:22px 28px}
.er-panel[hidden]{display:none}
.card{background:var(--card);border:1px solid var(--border);border-radius:6px;padding:18px 20px;margin-bottom:18px;min-width:0}
.card h2{font:20px/1.3 Georgia,serif;margin:0 0 12px}
.card h3{font-size:11px;letter-spacing:.08em;text-transform:uppercase;color:var(--muted);margin:18px 0 8px}
.grid2{display:grid;grid-template-columns:320px minmax(0,1fr);gap:18px;align-items:start}
.grid-half{display:grid;grid-template-columns:repeat(auto-fit,minmax(340px,1fr));gap:18px}
.filters{display:flex;gap:8px;flex-wrap:wrap;margin-bottom:12px}
.filters input,.filters select{font:inherit;padding:7px 10px;border:1px solid var(--border);border-radius:4px;background:#fff;color:var(--ink);min-width:0}
.filters input{flex:1 1 220px}
.ent-list{max-height:70vh;overflow:auto;margin:0;padding:0;list-style:none}
.ent-list li{display:flex;justify-content:space-between;gap:8px;padding:7px 8px;border-radius:4px;cursor:pointer}
.ent-list li:hover{background:var(--wash)}
.ent-list li[aria-selected=true]{background:var(--wash);font-weight:600}
.ent-list small{color:var(--muted);font-variant-numeric:tabular-nums}
table{border-collapse:collapse;width:100%;font-size:12px}
th{text-align:left;font-weight:600;color:var(--muted);padding:6px 8px;border-bottom:1px solid var(--border);white-space:nowrap}
td{padding:7px 8px;border-bottom:1px solid #eef2ec;vertical-align:top}
td.num,th.num{text-align:right;font-variant-numeric:tabular-nums}
.table-wrap{overflow-x:auto}
.chip{display:inline-block;padding:1px 7px;border-radius:10px;font-size:11px;color:#fff;white-space:nowrap}
.chip.very_high{background:var(--very_high)}.chip.high{background:var(--high)}.chip.medium{background:var(--medium)}.chip.low{background:var(--low)}
.act{font-size:11px;color:var(--muted);white-space:nowrap}
.act.human_review{color:#8a5d14;font-weight:600}
.bar{height:6px;border-radius:3px;background:#e8ede6;min-width:60px;position:relative}
.bar i{position:absolute;inset:0 auto 0 0;border-radius:3px}
.feat{display:grid;grid-template-columns:repeat(5,34px);gap:2px;font-size:10px;color:var(--muted);font-variant-numeric:tabular-nums}
.feat span{text-align:center}
.sent{font-family:Georgia,serif;line-height:1.55;color:#465149;max-width:560px}
.sent mark{background:#f2e6c4;color:inherit;padding:0 2px;border-radius:2px}
.cite{color:var(--muted);font-size:11px;white-space:nowrap}
tr.row{cursor:pointer}
tr.row:hover td{background:#fafcf8}
tr.detail td{background:#f8faf7}
.pager{display:flex;gap:10px;align-items:center;justify-content:flex-end;margin-top:10px;font-size:12px;color:var(--muted)}
.pager button{font:inherit;padding:5px 10px;border:1px solid var(--border);background:#fff;border-radius:4px;cursor:pointer}
.pager button:disabled{opacity:.4;cursor:default}
.hbar{display:grid;grid-template-columns:120px minmax(0,1fr) 70px;gap:10px;align-items:center;margin:6px 0;font-size:12px}
.hbar .bar{height:14px}
.note{color:var(--muted);font-size:12px;line-height:1.6}
.muted{color:var(--muted)}
.nowrap{white-space:nowrap}
.big{font:30px/1.2 Georgia,serif;margin:0}
.delta-up{color:var(--accent)}.delta-down{color:var(--low)}
@media(max-width:900px){.grid2{grid-template-columns:minmax(0,1fr)}.ent-list{max-height:260px}.er-panel,.er-banner{padding-left:16px;padding-right:16px}.er-tabs{padding:8px 16px 0}.er-stats{display:none}}
</style>
</head><body>
<header class="topbar"><a class="wordmark" href="er.html">Mahabharata<span>ENTITY NORMALIZATION</span></a><div class="er-stats" id="stats"></div></header>
<div class="er-banner" id="banner"></div>
<nav class="er-tabs" role="tablist">
<button role="tab" data-tab="characters" aria-selected="true">Characters</button>
<button role="tab" data-tab="mentions" aria-selected="false">Mentions</button>
<button role="tab" data-tab="review" aria-selected="false">Review queue</button>
<button role="tab" data-tab="overview" aria-selected="false">Bands &amp; gate</button>
<button role="tab" data-tab="gold" aria-selected="false">Gold standard</button>
</nav>

<section class="er-panel" id="tab-characters">
<div class="grid2">
<div class="card"><div class="filters"><input id="ent-q" placeholder="Find a character or alias" aria-label="Find a character or alias"></div><ul class="ent-list" id="ent-list"></ul></div>
<div class="card" id="ent-detail"></div>
</div>
</section>

<section class="er-panel" id="tab-mentions" hidden>
<div class="card">
<div class="filters">
<input id="m-q" placeholder="Search name, candidate or sentence" aria-label="Search mentions">
<select id="m-band" aria-label="Band"><option value="">All bands</option></select>
<select id="m-action" aria-label="Action"><option value="">All actions</option></select>
<select id="m-parva" aria-label="Parva"><option value="">All parvas</option></select>
<select id="m-multi" aria-label="Candidates"><option value="">Any candidates</option><option value="1">More than one candidate</option></select>
</div>
<div class="table-wrap"><table id="m-table"></table></div>
<div class="pager" id="m-pager"></div>
</div>
</section>

<section class="er-panel" id="tab-review" hidden>
<div class="grid2">
<div class="card"><h2>By name</h2><p class="note">Mentions routed to a human: a medium score, or a runner-up too close to call.</p><ul class="ent-list" id="rv-list"></ul></div>
<div class="card"><div class="table-wrap"><table id="rv-table"></table></div><div class="pager" id="rv-pager"></div></div>
</div>
</section>

<section class="er-panel" id="tab-overview" hidden></section>
<section class="er-panel" id="tab-gold" hidden></section>

<script>
// Runs after the deferred er-data.js, the way explorer.js runs after projection.js.
document.addEventListener("DOMContentLoaded", function(){
"use strict";
const D = window.KGER;
if (!D) { document.body.insertAdjacentHTML("beforeend", "<p class='er-panel'>er-data.js did not load. Keep it in the same folder as this page.</p>"); return; }
const BANDS = D.bands.map(b => b[1]), ACTIONS = D.actions;
const esc = s => String(s == null ? "" : s).replace(/[&<>"']/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"})[c]);
const fmt = n => Number(n).toLocaleString("en");
const pct = x => (100 * x).toFixed(1) + "%";
const bandOf = c => { for (const [lo, name] of D.bands) if (c >= lo) return name; return "low"; };
const $ = id => document.getElementById(id);

// Mention rows: [id, surface, parva, adhyaya, sentence index, action index, candidates]
// candidates: [name, confidence, A, C, R, S, E, sources], best first.
const M = D.mentions.map(r => ({id: r[0], surface: r[1], parva: r[2], adhyaya: r[3], sent: D.sentences[r[4]],
  action: ACTIONS[r[5]], cands: r[6], top: r[6][0], band: bandOf(r[6][0][1]),
  margin: r[6].length > 1 ? r[6][0][1] - r[6][1][1] : null}));

// --- header, banner
$("stats").innerHTML = [["mentions scored", D.summary.scored], ["characters", D.entities.length],
  ["auto-accepted", D.summary.actions.auto_accept || 0], ["to review", D.summary.actions.human_review || 0]]
  .map(([k, v]) => "<div><b>" + fmt(v) + "</b>" + k + "</div>").join("");
$("banner").innerHTML = "<strong>Proposals only.</strong> Nothing on this page is in the asserted graph. " +
  "Confidence = " + Object.entries(D.weights).map(([k, v]) => v.toFixed(2) + "&middot;" + k).join(" + ") +
  " (" + (D.gold ? "learned weights are reported under Gold standard" : "the design document's weights, not learned") + "). " +
  (D.gold ? "Calibration is measured against " + fmt(D.gold.n) + " hand-linked mentions."
          : "Confidence is not accuracy: no gold standard has been scored yet.");

// --- tabs, linkable as er.html#mentions
function showTab(name) {
  const b = document.querySelector(".er-tabs button[data-tab='" + name + "']");
  if (!b) return;
  document.querySelectorAll(".er-tabs button").forEach(x => x.setAttribute("aria-selected", x === b));
  document.querySelectorAll(".er-panel").forEach(p => p.hidden = p.id !== "tab-" + name);
}
document.querySelectorAll(".er-tabs button").forEach(b => b.addEventListener("click", () => {
  showTab(b.dataset.tab);
  history.replaceState(null, "", "#" + b.dataset.tab);
}));
showTab(location.hash.slice(1) || "characters");

// --- shared renderers
function sentenceHTML(s, surface) {
  // Whole words only: "Nara" must not light up inside "Narayana".
  const t = esc(s), m = esc(surface);
  const re = new RegExp("(^|[^A-Za-z])(" + m.replace(/[.*+?^${}()|[\]\\]/g, "\\$&") + ")(?![A-Za-z])");
  return t.replace(re, "$1<mark>$2</mark>");
}
function confCell(c) {
  const b = bandOf(c);
  return "<span class='chip " + b + "'>" + c.toFixed(2) + "</span>";
}
function featHTML(c) {
  return "<div class='feat' title='A alias, C context, R relations, S source, E type'>" +
    ["A", "C", "R", "S", "E"].map((k, i) => "<span title='" + k + "'>" + k + " " + c[2 + i].toFixed(2) + "</span>").join("") + "</div>";
}
function candTable(m) {
  return "<table><tr><th>candidate</th><th class='num'>confidence</th><th>A &middot; C &middot; R &middot; S &middot; E</th><th>proposed by</th></tr>" +
    m.cands.map(c => "<tr><td>" + esc(c[0]) + "</td><td class='num'>" + confCell(c[1]) + "</td><td>" + featHTML(c) +
      "</td><td class='muted'>" + esc(c[7]) + "</td></tr>").join("") + "</table>";
}
function mentionTable(tableEl, pagerEl, rows, page, onPage) {
  const per = 50, pages = Math.max(1, Math.ceil(rows.length / per));
  page = Math.min(page, pages - 1);
  const slice = rows.slice(page * per, page * per + per);
  tableEl.innerHTML = "<thead><tr><th>mention</th><th>linked to</th><th class='num'>confidence</th><th>action</th><th>runner-up</th><th>scores</th><th>passage</th></tr></thead><tbody>" +
    slice.map((m, i) => "<tr class='row' data-i='" + i + "'><td><strong>" + esc(m.surface) + "</strong></td><td>" + esc(m.top[0]) +
      "</td><td class='num'>" + confCell(m.top[1]) + "</td><td class='act " + m.action + "'>" + m.action.replace(/_/g, " ") +
      "</td><td class='muted'>" + (m.cands[1] ? esc(m.cands[1][0]) + " <span class='cite'>(&minus;" + m.margin.toFixed(2) + ")</span>" : "&mdash;") +
      "</td><td>" + featHTML(m.top) + "</td><td><div class='sent'>" + sentenceHTML(m.sent, m.surface) +
      "</div><div class='cite'>" + esc(m.parva) + (m.adhyaya != null ? " / " + m.adhyaya : "") + "</div></td></tr>").join("") + "</tbody>";
  tableEl.querySelectorAll("tr.row").forEach(tr => tr.addEventListener("click", () => {
    const next = tr.nextElementSibling;
    if (next && next.classList.contains("detail")) { next.remove(); return; }
    tr.insertAdjacentHTML("afterend", "<tr class='detail'><td colspan='7'>" + candTable(slice[+tr.dataset.i]) + "</td></tr>");
  }));
  pagerEl.innerHTML = "<span>" + fmt(rows.length) + " mentions</span><button " + (page ? "" : "disabled") + ">Previous</button><span>page " +
    (page + 1) + " of " + pages + "</span><button " + (page < pages - 1 ? "" : "disabled") + ">Next</button>";
  const [prev, next] = pagerEl.querySelectorAll("button");
  prev.onclick = () => onPage(page - 1); next.onclick = () => onPage(page + 1);
}

// --- characters
const byEntity = {};
M.forEach(m => { if (m.action === "auto_accept" || m.action === "accept_sample_verify") (byEntity[m.top[0]] = byEntity[m.top[0]] || []).push(m); });
let entSel = D.entities.length ? D.entities[0].name : null;
function renderEntList() {
  const q = $("ent-q").value.trim().toLowerCase();
  const list = D.entities.filter(e => !q || e.name.toLowerCase().includes(q) || e.aliases.some(a => a[0].toLowerCase().includes(q)));
  $("ent-list").innerHTML = list.slice(0, 600).map(e => "<li role='option' aria-selected='" + (e.name === entSel) + "' data-n='" + esc(e.name) +
    "'><span>" + esc(e.name) + " <small>" + esc(e.id) + "</small></span><small>" + fmt(e.mentions) + "</small></li>").join("") ||
    "<li class='muted'>No match</li>";
  $("ent-list").querySelectorAll("li[data-n]").forEach(li => li.onclick = () => { entSel = li.dataset.n; renderEntList(); renderEntity(); });
}
function renderEntity() {
  const e = D.entities.find(x => x.name === entSel);
  if (!e) { $("ent-detail").innerHTML = "<p class='muted'>No accepted links yet.</p>"; return; }
  const ms = byEntity[e.name] || [];
  $("ent-detail").innerHTML = "<p class='cite'>" + esc(e.id) + " &middot; canonical entity</p><h1 class='big'>" + esc(e.name) + "</h1>" +
    "<p class='note'>" + fmt(e.mentions) + " accepted mentions under " + e.aliases.length + " surface forms. Aliases are the names these mentions were written with; " +
    "a name shared with another character appears under each, split by passage.</p>" +
    "<h3>Aliases</h3><div class='table-wrap'><table><tr><th>written as</th><th class='num'>mentions</th><th class='num'>mean confidence</th><th>proposed by</th></tr>" +
    e.aliases.map(a => "<tr><td>" + esc(a[0]) + "</td><td class='num'>" + fmt(a[1]) + "</td><td class='num'>" + confCell(a[2]) +
      "</td><td class='muted'>" + esc(a[3]) + "</td></tr>").join("") + "</table></div>" +
    "<h3>Accepted mentions</h3><div class='table-wrap'><table id='ent-m'></table></div><div class='pager' id='ent-p'></div>";
  const go = p => mentionTable($("ent-m"), $("ent-p"), ms, p, go); go(0);
}
$("ent-q").addEventListener("input", renderEntList);
renderEntList(); renderEntity();

// --- mentions
const parvas = [...new Set(M.map(m => m.parva))].sort();
BANDS.forEach(b => $("m-band").insertAdjacentHTML("beforeend", "<option>" + b + "</option>"));
ACTIONS.forEach(a => $("m-action").insertAdjacentHTML("beforeend", "<option value='" + a + "'>" + a.replace(/_/g, " ") + "</option>"));
parvas.forEach(p => $("m-parva").insertAdjacentHTML("beforeend", "<option>" + esc(p) + "</option>"));
let mRows = M;
function filterMentions() {
  const q = $("m-q").value.trim().toLowerCase(), b = $("m-band").value, a = $("m-action").value,
        p = $("m-parva").value, multi = $("m-multi").value;
  mRows = M.filter(m => (!b || m.band === b) && (!a || m.action === a) && (!p || m.parva === p) &&
    (!multi || m.cands.length > 1) &&
    (!q || m.surface.toLowerCase().includes(q) || m.top[0].toLowerCase().includes(q) || m.sent.toLowerCase().includes(q)));
  goM(0);
}
const goM = p => mentionTable($("m-table"), $("m-pager"), mRows, p, goM);
["m-q", "m-band", "m-action", "m-parva", "m-multi"].forEach(id => $(id).addEventListener("input", filterMentions));
goM(0);

// --- review queue
const review = M.filter(m => m.action === "human_review");
const rvCount = {};
review.forEach(m => rvCount[m.surface] = (rvCount[m.surface] || 0) + 1);
let rvSel = null;
function renderReview() {
  $("rv-list").innerHTML = "<li aria-selected='" + (rvSel === null) + "' data-n=''><span>All names</span><small>" + fmt(review.length) + "</small></li>" +
    Object.entries(rvCount).sort((a, b) => b[1] - a[1]).map(([s, n]) => "<li aria-selected='" + (s === rvSel) + "' data-n='" + esc(s) +
      "'><span>" + esc(s) + "</span><small>" + fmt(n) + "</small></li>").join("");
  $("rv-list").querySelectorAll("li").forEach(li => li.onclick = () => { rvSel = li.dataset.n || null; renderReview(); });
  const rows = rvSel ? review.filter(m => m.surface === rvSel) : review;
  const go = p => mentionTable($("rv-table"), $("rv-pager"), rows, p, go); go(0);
}
renderReview();

// --- bands and gate
function hbars(obj, order, total, cls) {
  return order.filter(k => obj[k] != null).map(k => "<div class='hbar'><span>" + esc(k.replace(/_/g, " ")) + "</span><div class='bar'><i class='" + (cls ? k : "") +
    "' style='width:" + (100 * obj[k] / total).toFixed(1) + "%;background:" + (cls ? "var(--" + k + ")" : "var(--accent)") + "'></i></div><span class='num'>" +
    fmt(obj[k]) + "</span></div>").join("");
}
const g = D.gate, S = D.summary;
const gateRow = (label, k, req, kind) => {
  const b = g.before[k], a = g.after[k];
  const show = v => kind === "pct" ? pct(v) : kind === "dec" ? v.toFixed(2) : (typeof v === "number" ? fmt(v) : esc(v));
  const cls = typeof a === "number" && typeof b === "number" && a !== b ? (a > b ? "delta-up" : "delta-down") : "";
  return "<tr><td class='nowrap'>" + label + "</td><td class='num'>" + show(b) + "</td><td class='num " + cls + "'>" + show(a) + "</td><td class='muted nowrap'>" + req + "</td></tr>";
};
$("tab-overview").innerHTML = "<div class='grid-half'>" +
  "<div class='card'><h2>Confidence bands</h2>" + hbars(S.bands, BANDS, S.scored, true) +
  "<p class='note'>&ge; " + D.bands[0][0] + " auto-accept, &ge; " + D.bands[1][0] + " accept with sample verification, &ge; " + D.bands[2][0] +
  " human review, below that reject.</p><h3>Actions</h3>" + hbars(S.actions, ACTIONS, S.scored, false) +
  "<p class='note'>" + fmt(S.multi) + " mentions had more than one candidate. " + fmt(S.close) + " went to review because the runner-up was within " +
  D.margin + ", whatever their score.</p></div>" +
  "<div class='card'><h2>Effect on the Phase 6 entry gate</h2><div class='table-wrap'><table><tr><th>measure</th><th class='num'>before</th><th class='num'>after</th><th>required</th></tr>" +
  gateRow("distinct triples", "triples", "&ge; 10,000") + gateRow("entities", "entities", "") +
  gateRow("entities of degree &ge; 5", "degree_ge_5", "&ge; 500") + gateRow("largest relation share", "top_share", "&le; 40%", "pct") +
  gateRow("mean degree", "mean_degree", "", "dec") + gateRow("highest degree", "max_degree", "") + "</table></div>" +
  "<p class='note'>Relation endpoints rewritten through accepted links only. Merging turns Shakra, Vasava and Maghavan into Indra: a triple written under two names becomes one, " +
  "a triple between two names of one person is dropped, and several nodes of degree 5 become one node. Normalization cannot add a triple.</p></div></div>" +
  "<div class='card'><h2>Ambiguous names, how their mentions split</h2><div class='table-wrap'><table><tr><th>written as</th><th class='num'>mentions</th><th>accepted as</th><th class='num'>to review</th></tr>" +
  D.ambiguous.map(a => "<tr><td>" + esc(a.surface) + "</td><td class='num'>" + fmt(a.mentions) + "</td><td>" +
    (Object.entries(a.accepted).map(([k, v]) => esc(k) + " " + fmt(v)).join(", ") || "<span class='muted'>&mdash;</span>") +
    "</td><td class='num'>" + fmt(a.review) + "</td></tr>").join("") + "</table></div></div>";

// --- gold standard
function relTable(rows) {
  return "<table><tr><th>confidence</th><th class='num'>mentions</th><th class='num'>mean confidence</th><th class='num'>share correct</th><th>gap</th></tr>" +
    rows.map(r => "<tr><td>" + r[0] + "</td><td class='num'>" + fmt(r[1]) + "</td><td class='num'>" + r[2].toFixed(3) + "</td><td class='num'>" +
      r[3].toFixed(3) + "</td><td><div class='bar'><i style='width:" + Math.min(100, 100 * Math.abs(r[2] - r[3])).toFixed(1) +
      "%;background:var(--" + (Math.abs(r[2] - r[3]) > .1 ? "low" : "high") + ")'></i></div></td></tr>").join("") + "</table>";
}
const G = D.gold;
$("tab-gold").innerHTML = !G ? "<div class='card'><h2>No gold standard yet</h2><p class='note'>Fill <code>gold_template.csv</code> (the mention is marked in the sentence; " +
  "write one of the listed candidates, OTHER, NOT_A_PERSON or UNSURE in <code>correct_entity</code>), save it as <code>er_gold.csv</code> and run again. " +
  "This tab then shows precision, recall and F1 per threshold, calibration, and the weights learned from your links. Do not open <code>gold_key.csv</code> first.</p></div>" :
  "<div class='card'><h2>Against " + fmt(G.n) + " hand-linked mentions</h2><div class='table-wrap'><table><tr><th>accept at</th><th class='num'>predicted</th><th class='num'>TP</th><th class='num'>FP</th><th class='num'>FN</th><th class='num'>precision</th><th class='num'>recall</th><th class='num'>F1</th></tr>" +
  G.thresholds.map(t => "<tr><td>" + t[0].toFixed(2) + "</td>" + t.slice(1, 5).map(v => "<td class='num'>" + fmt(v) + "</td>").join("") +
    t.slice(5).map(v => "<td class='num'>" + v.toFixed(3) + "</td>").join("") + "</tr>").join("") + "</table></div></div>" +
  "<div class='grid-half'>" + G.calibration.map(c => "<div class='card'><h2>Calibration, " + esc(c.name) + "</h2><p class='note'>ECE " + c.ece.toFixed(3) +
    ", Brier " + c.brier.toFixed(3) + ". A calibrated score has share correct close to mean confidence in every row.</p><div class='table-wrap'>" + relTable(c.rows) + "</div></div>").join("") + "</div>" +
  (G.coef ? "<div class='card'><h2>Learned weights</h2><p class='note'>Logistic regression over every (mention, candidate) row, cross-validated by name so no alias is in both train and test.</p><div class='table-wrap'><table><tr><th>feature</th><th class='num'>coefficient</th><th class='num'>share of positive weight</th><th class='num'>design weight</th></tr>" +
    G.coef.map(r => "<tr><td>" + r[0] + "</td><td class='num'>" + r[1].toFixed(3) + "</td><td class='num'>" + r[2].toFixed(2) + "</td><td class='num'>" + r[3].toFixed(2) + "</td></tr>").join("") +
    "</table></div></div>" : "");
});
</script>
</body></html>

In [ ]:
# Cell 5: Phases 0 to 2, then the confidence scoring.
# ENCODER: the sentence encoder for C. "tfidf" needs no download and no GPU but
# separates contexts less well.
import subprocess, time, os, shutil

ENCODER = "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none, the encoder runs on CPU")

STEPS = [
    ("src/build_spine.py", []),        # chapter boundaries from parva_pdfs
    ("src/phase1_mentions.py", []),    # mentions.parquet with (parva, adhyaya) citations
    ("src/phase2_entities.py", []),    # apposition candidates and the corpus prior
    ("src/phase2b_confidence.py", ["--encoder", ENCODER]),
]
env = dict(os.environ, PYTHONUNBUFFERED="1")
for script, args in STEPS:
    print("\n==== %s %s" % (script, " ".join(args)), flush=True)
    t = time.time()
    p = subprocess.Popen([sys.executable, script] + args, cwd=WORK, env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    code = p.wait()
    print("---- exit %d, %.1f min" % (code, (time.time() - t) / 60), flush=True)
    if code:
        raise SystemExit("%s failed with exit %d." % (script, code))

In [ ]:
# Cell 6: the report, and a look at the hardest cases.
import pandas as pd
from IPython.display import Markdown, display
ER = WORK / "build/er"
display(Markdown((ER / "er_report.md").read_text(encoding="utf-8")))
links = pd.read_csv(ER / "mention_links.csv")
print("Mentions sent to human review, most frequent surfaces:")
display(links[links.action == "human_review"].surface.value_counts().head(20))
print("Sample of close calls (runner-up within the margin):")
display(links[links.margin < 0.10][["surface", "candidate", "runner_up", "confidence",
                                    "margin", "A", "C", "R", "S", "E", "sentence"]].head(15))

In [ ]:
# Cell 7: bundle the results for download from the Output tab.
# The CSVs carry verbatim sentences of a copyrighted translation; keep this private.
import zipfile
dest = Path("/kaggle/working/mahabharata_er_outputs.zip")
with zipfile.ZipFile(dest, "w", zipfile.ZIP_DEFLATED) as z:
    for p in (WORK / "build/er").glob("*"):
        z.write(p, p.relative_to(WORK))
    for rel in ["build/phase1_report.md", "build/appellations.csv", "build/phase2_resolution.json"]:
        if (WORK / rel).exists():
            z.write(WORK / rel, rel)
print(dest, "%.1f MB" % (dest.stat().st_size / 1e6))

# The page as one file, at the top of the Output tab: styles and data are inside
# it, so this single download opens in any browser with nothing else beside it.
html = Path("/kaggle/working/mahabharata_er.html")
shutil.copy(WORK / "build/er/mahabharata_er.html", html)
print(html, "%.1f MB" % (html.stat().st_size / 1e6))
from IPython.display import FileLink, display
display(FileLink(str(html.relative_to("/kaggle/working"))))

## Opening the page

**Quickest:** download **`mahabharata_er.html`** from the Output tab (or the link Cell 7 prints) and open it in a browser. It is one file with the styles and data inside, about 12 MB.

The zip carries the same page as three files: unzip `mahabharata_er_outputs.zip` and open `build/er/er.html`. It is built like the explorer's `out/index.html`: a fixed page with its data in `er-data.js` beside it, so it runs from disk with no server and sends nothing anywhere. Keep `er.html`, `er-data.js` and `explorer.css` in the same folder. Tabs: **Characters** (canonical IDs, aliases, accepted mentions), **Mentions** (every scored mention, filterable, click a row for all candidates and their A·C·R·S·E), **Review queue**, **Bands & gate**, **Gold standard**.

## After the first run: building the gold standard

1. Download `build/er/gold_template.csv` from the zip. **Don't open `gold_key.csv` first**: it holds the system's answers, and seeing them anchors your judgement.
2. For each row, read the `sentence` (the mention is in `[[ ]]`) and write in `correct_entity`:
   - one of the names in `candidates`, spelled exactly as listed, or
   - `OTHER` if it's a person not in the list, or
   - `NOT_A_PERSON`, or
   - `UNSURE` (dropped from scoring).
3. Save it as `er_gold.csv`, upload it as a new private Kaggle dataset, **Add Data** it to this notebook, and run all cells again. The report gains precision, recall, F1, calibration and the learned weights. `candidate_features_learned.csv` holds every candidate rescored with the learned model.

The document suggests 1,000 mentions. Scoring works with fewer, but the learned weights need at least a few hundred rows over many different names to mean anything.